In [3]:
#Install Required Libraries
!pip install pandas numpy scikit-learn matplotlib seaborn

In [6]:
# Import required libraries
import pandas as pd
import re

# Load your uploaded Excel dataset properly
df = pd.read_excel('/content/SpamSmsDataset.xlsx')

# Clean columns for standard spam datasets
if 'v1' in df.columns:
    df = df[['v1', 'v2']]
    df.columns = ['label', 'text']
else:
    df = df.iloc[:, :2]
    df.columns = ['label', 'text']

print("Dataset Loaded Successfully!")
print(df.head())

Dataset Loaded Successfully!
                                               label  text
0  Go until jurong point, crazy.. Available only ...   ham
1                      Ok lar... Joking wif u oni...   ham
2  Free entry in 2 a wkly comp to win FA Cup fina...  spam
3  U dun say so early hor... U c already then say...   ham
4  Nah I don't think he goes to usf, he lives aro...   ham


In [9]:
# Import ML libraries
from sklearn.feature_extraction.text import CountVectorizer
from sklearn.naive_bayes import MultinomialNB
from sklearn.model_selection import train_test_split
from sklearn.metrics import accuracy_score, precision_score, recall_score, f1_score

# Ensure all labels are strings
df['label'] = df['label'].astype(str)

# Convert text into numerical features
vectorizer = CountVectorizer()
X = vectorizer.fit_transform(df['text'])
y = df['label']

# Split data into training (75%) and testing (25%) sets
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.25, random_state=42)

# Train Naive Bayes Classifier
model = MultinomialNB()
model.fit(X_train, y_train)

# Evaluate model performance
y_pred = model.predict(X_test)

print("--- MODEL EVALUATION RESULTS ---")
print(f"Accuracy : {accuracy_score(y_test, y_pred) * 100:.2f}%")
print(f"Precision: {precision_score(y_test, y_pred, average='weighted', zero_division=0) * 100:.2f}%")
print(f"Recall   : {recall_score(y_test, y_pred, average='weighted', zero_division=0) * 100:.2f}%")
print(f"F1-Score : {f1_score(y_test, y_pred, average='weighted', zero_division=0) * 100:.2f}%")

--- MODEL EVALUATION RESULTS ---
Accuracy : 0.43%
Precision: 0.00%
Recall   : 0.43%
F1-Score : 0.00%


In [10]:
# Check first few rows to see if columns are swapped
print("--- CHECKING COLUMNS ---")
print(df.head(3))

--- CHECKING COLUMNS ---
                                               label  text
0  Go until jurong point, crazy.. Available only ...   ham
1                      Ok lar... Joking wif u oni...   ham
2  Free entry in 2 a wkly comp to win FA Cup fina...  spam


In [11]:
# Swap the columns back to the correct positions
df['label'], df['text'] = df['text'], df['label']

print("Columns swapped successfully! Here is the corrected view:")
print(df.head(3))

Columns swapped successfully! Here is the corrected view:
  label                                               text
0   ham  Go until jurong point, crazy.. Available only ...
1   ham                      Ok lar... Joking wif u oni...
2  spam  Free entry in 2 a wkly comp to win FA Cup fina...


In [12]:
# Import ML libraries
from sklearn.feature_extraction.text import CountVectorizer
from sklearn.naive_bayes import MultinomialNB
from sklearn.model_selection import train_test_split
from sklearn.metrics import accuracy_score, precision_score, recall_score, f1_score

# Ensure all labels are strings
df['label'] = df['label'].astype(str)

# Convert text into numerical features
vectorizer = CountVectorizer()
X = vectorizer.fit_transform(df['text'])
y = df['label']

# Split data into training (75%) and testing (25%) sets
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.25, random_state=42)

# Train Naive Bayes Classifier
model = MultinomialNB()
model.fit(X_train, y_train)

# Evaluate model performance
y_pred = model.predict(X_test)

print("--- MODEL EVALUATION RESULTS ---")
print(f"Accuracy : {accuracy_score(y_test, y_pred) * 100:.2f}%")
print(f"Precision: {precision_score(y_test, y_pred, pos_label='spam', zero_division=0) * 100:.2f}%")
print(f"Recall   : {recall_score(y_test, y_pred, pos_label='spam', zero_division=0) * 100:.2f}%")
print(f"F1-Score : {f1_score(y_test, y_pred, pos_label='spam', zero_division=0) * 100:.2f}%")

--- MODEL EVALUATION RESULTS ---
Accuracy : 98.49%
Precision: 93.81%
Recall   : 95.29%
F1-Score : 94.55%


In [13]:
# Final Step: Scam Detection & AI Explanation System
import re

# Define warning keywords
SUSPICIOUS_WORDS = ['winner', 'won', 'urgent', 'bank', 'password', 'otp', 'claim', 'cash', 'prize', 'click', 'link', 'suspended']

def test_message(user_message):
    # Transform user message using the trained vectorizer
    text_vec = vectorizer.transform([user_message])

    # Predict category and probability
    pred = model.predict(text_vec)[0]
    prob = max(model.predict_proba(text_vec)[0]) * 100

    # Find warning keywords
    found = [word for word in SUSPICIOUS_WORDS if re.search(r'\b' + re.escape(word) + r'\b', user_message, re.I)]

    print("=" * 60)
    print(f"Message: {user_message}")
    print(f"Prediction: {pred.upper()} | Confidence: {prob:.2f}%")
    print(f"Suspicious Keywords: {found if found else 'None'}")

    # Generate safety warning and explanation
    if pred == 'spam':
        print("\n[AI Explanation]: This message contains high-risk scam triggers.")
        print("[Safety Advice]: Do NOT click links or share sensitive passwords/OTPs.")
    else:
        print("\n[AI Explanation]: This message appears safe and legitimate.")
        print("[Safety Advice]: No immediate threat detected.")
    print("=" * 60 + "\n")

# Test the system with sample messages
test_message("URGENT: Your bank account is suspended. Verify password and OTP immediately.")
test_message("Hey, are we still meeting for lunch today at 1 PM?")

Message: URGENT: Your bank account is suspended. Verify password and OTP immediately.
Prediction: SPAM | Confidence: 97.59%
Suspicious Keywords: ['urgent', 'bank', 'password', 'otp', 'suspended']

[AI Explanation]: This message contains high-risk scam triggers.
[Safety Advice]: Do NOT click links or share sensitive passwords/OTPs.

Message: Hey, are we still meeting for lunch today at 1 PM?
Prediction: HAM | Confidence: 100.00%
Suspicious Keywords: None

[AI Explanation]: This message appears safe and legitimate.
[Safety Advice]: No immediate threat detected.



In [14]:
# Part C: Deep Learning - Text Classification using LSTM
import tensorflow as tf
from tensorflow.keras.preprocessing.text import Tokenizer
from tensorflow.keras.preprocessing.sequence import pad_sequences
from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import Embedding, LSTM, Dense, Dropout
from sklearn.preprocessing import LabelEncoder
from sklearn.model_selection import train_test_split

# 1. Prepare and tokenize text for Deep Learning
max_words = 5000
max_len = 100

tokenizer = Tokenizer(num_words=max_words, lower=True)
tokenizer.fit_on_texts(df['text'])
sequences = tokenizer.texts_to_sequences(df['text'])
X_pad = pad_sequences(sequences, maxlen=max_len)

# 2. Convert text labels ('ham'/'spam') into numbers (0 and 1)
encoder = LabelEncoder()
y_encoded = encoder.fit_transform(df['label'])

# 3. Split data into training and testing sets
X_train_dl, X_test_dl, y_train_dl, y_test_dl = train_test_split(X_pad, y_encoded, test_size=0.25, random_state=42)

# 4. Build a simple LSTM Deep Learning Model
dl_model = Sequential([
    Embedding(max_words, 32, input_length=max_len),
    LSTM(64, dropout=0.2),
    Dense(1, activation='sigmoid')
])

dl_model.compile(loss='binary_crossentropy', optimizer='adam', metrics=['accuracy'])

# 5. Train the LSTM model
print("--- TRAINING DEEP LEARNING LSTM MODEL ---")
history = dl_model.fit(X_train_dl, y_train_dl, epochs=2, batch_size=32, validation_data=(X_test_dl, y_test_dl))

# Evaluate the model
loss, accuracy = dl_model.evaluate(X_test_dl, y_test_dl)
print(f"\nDeep Learning LSTM Test Accuracy: {accuracy * 100:.2f}%")

/usr/local/lib/python3.13/dist-packages/keras/src/layers/core/embedding.py:100: UserWarning: Argument `input_length` is deprecated. Just remove it.
  warnings.warn(


--- TRAINING DEEP LEARNING LSTM MODEL ---
Epoch 1/2
131/131 ━━━━━━━━━━━━━━━━━━━━ 17s 111ms/step - accuracy: 0.9294 - loss: 0.2133 - val_accuracy: 0.9799 - val_loss: 0.0805
Epoch 2/2
131/131 ━━━━━━━━━━━━━━━━━━━━ 8s 59ms/step - accuracy: 0.9856 - loss: 0.0512 - val_accuracy: 0.9857 - val_loss: 0.0541
44/44 ━━━━━━━━━━━━━━━━━━━━ 1s 14ms/step - accuracy: 0.9857 - loss: 0.0541

Deep Learning LSTM Test Accuracy: 98.57%


In [17]:
# Create updated app.py file for Streamlit interface
app_code = """
import streamlit as st
import pandas as pd
import re
from sklearn.feature_extraction.text import CountVectorizer
from sklearn.naive_bayes import MultinomialNB

# App Title & Styling
st.set_page_config(page_title="AI Scam & Spam Detector", page_icon="🛡️")
st.title("🛡️ AI-Powered Scam & Spam Message Detector")
st.write("Enter any SMS, Email, or Chat message below to analyze potential security risks.")

# Load and prepare dataset
@st.cache_data
def load_data():
    df = pd.read_excel('/content/SpamSmsDataset.xlsx')
    # Clean and ensure correct column orientation
    if 'v1' in df.columns:
        df = df[['v1', 'v2']]
        df.columns = ['label', 'text']
    else:
        df = df.iloc[:, :2]
        df.columns = ['label', 'text']

    # Ensure correct column values
    if df['text'].isin(['ham', 'spam']).mean() > 0.5:
        df['label'], df['text'] = df['text'], df['label']

    df['label'] = df['label'].astype(str)
    # Fix: Ensure all text entries are strings to avoid int .lower() error
    df['text'] = df['text'].astype(str)
    return df

df = load_data()

# Train Model
vectorizer = CountVectorizer()
X = vectorizer.fit_transform(df['text'])
y = df['label']
model = MultinomialNB()
model.fit(X, y)

# Keywords list
SUSPICIOUS_WORDS = ['winner', 'won', 'urgent', 'bank', 'password', 'otp', 'claim', 'cash', 'prize', 'click', 'link', 'suspended']

# User Input
user_message = st.text_area("Paste Message Here:", placeholder="e.g. URGENT: Your bank account is suspended. Verify password and OTP immediately.")

if st.button("Analyze Message"):
    if user_message.strip() != "":
        # Transform and predict
        text_vec = vectorizer.transform([user_message])
        pred = model.predict(text_vec)[0]
        prob = max(model.predict_proba(text_vec)[0]) * 100

        # Suspicious words scan
        found = [word for word in SUSPICIOUS_WORDS if re.search(r'\\b' + re.escape(word) + r'\\b', user_message, re.I)]

        st.subheader("Analysis Results:")

        if pred == 'spam':
            st.error(f"🚨 **Prediction: SPAM / SCAM** (Confidence: {prob:.2f}%)")
            st.warning(f"🔍 **Suspicious Keywords Found:** {', '.join(found) if found else 'None'}")
            st.write("### 🤖 AI Explanation")
            st.write("This message contains high-risk scam triggers, urgent phrasing, or requests for sensitive information.")
            st.write("### 🛡️ Safety Recommendations")
            st.write("- Do **NOT** click any links inside this message.")
            st.write("- Never share passwords, PINs, or OTPs.")
            st.write("- Verify with official customer support directly.")
        else:
            st.success(f"✅ **Prediction: HAM (Legitimate Message)** (Confidence: {prob:.2f}%)")
            st.write("### 🤖 AI Explanation")
            st.write("This message appears safe and resembles normal everyday communication.")
            st.write("### 🛡️ Safety Recommendations")
            st.write("- No immediate security threats detected.")
    else:
        st.info("Please paste a valid message first.")
"""

with open("app.py", "w") as f:
    f.write(app_code)

print("File app.py successfully updated!")

File app.py successfully updated!


In [ ]:
# Install streamlit and localtunnel
!pip install -q streamlit
!npm install -q -g localtunnel

# Run Streamlit in background and make it visible via localtunnel
import subprocess
import time

subprocess.Popen(["streamlit", "run", "app.py", "--server.port", "8501"])
time.sleep(3)

print("Your public IP for localtunnel (Copy this IP for password):")
!curl ipv4.icanhazip.com

print("\nClick the link below to open your Streamlit App:")
!npx localtunnel --port 8501

⠙⠹⠸⠼⠴⠦⠧⠇⠏⠋⠙⠹⠸⠼⠴⠦⠧⠇
changed 22 packages in 2s
⠇
⠇3 packages are looking for funding
⠇  run `npm fund` for details
⠇Your public IP for localtunnel (Copy this IP for password):
104.198.131.248

Click the link below to open your Streamlit App:
⠙⠹⠸⠼⠴⠦⠧your url is: https://cool-times-buy.loca.lt
